# Navier–Stokes: error versus the output norm $H^\alpha$

We learn the map from a forcing $f$ to the vorticity $w(\cdot, T)$ of the 2-D incompressible Navier–Stokes equations on the torus (vorticity form, $w(0) = 0$, $\mathrm{Re} = 500$, $T = 5$), from a fixed dataset of 10000 training and 2000 test pairs on a $128 \times 128$ grid ([Zenodo 12825163](https://zenodo.org/records/12825163); see `raw_data/README.md`). We cannot choose where the inputs are, so the input measure is the empirical measure of the training forcings and we use `owls.EmpiricalWLS`, which samples optimally from the dataset.

* Inputs: PCA coefficients of the forcing that keep 99.5% of the variance (`encode_inputs`), scaled to $[-1, 1]$. Each coefficient gets a symmetric Jacobi measure with matching variance; it only selects the polynomial family (the basis is orthonormalized on the data).
* Outputs: Fourier coefficients in a basis orthonormal in $H^\alpha$, keeping 99.9% of the average $H^\alpha$ energy (`encode_outputs`).
* Approximation space: polynomial operators with an anisotropic hyperbolic cross index set, weights $w_j = \sigma_j / \sigma_1$ (normalized PCA singular values), univariate degree at most 10.

For $\alpha \in \{-5, -3, -1, 0, 1, 3, 5\}$ we fit the operator with the hyperbolic cross $k = 28$ from $M = \lfloor N\log N\rfloor$ optimally drawn training pairs, and report the average relative $H^\alpha$ test error (2000 test pairs), along with the output dimension $\dim \mathcal Y_h(\alpha)$ needed to keep 99.9% of the $H^\alpha$ energy.

**Status:** the data in `data/` are from the arXiv v1 run; this experiment was not rerun after the cleanup. Rerunning needs the dataset.

In [ ]:
RECOMPUTE = False  # True: rerun the experiment and overwrite data/ (see ../README.md for run times)
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig
from ns_utils import load_ns_data, encode_inputs, encode_outputs

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters (as run)
ALPHAS = [-5, -3, -1, 0, 1, 3, 5]
HC_K = 28
MAX_DEGREE = 10
N_MAX = 50000
SEED = 0
if QUICK:
    ALPHAS, HC_K = [-1, 1], 4

In [ ]:
if RECOMPUTE or QUICK:
    X_train, Y_train, X_test, Y_test = load_ns_data("raw_data")
    if QUICK:
        X_train, Y_train, X_test, Y_test = X_train[:600], Y_train[:600], X_test[:100], Y_test[:100]
    x_train, x_test, pca = encode_inputs(X_train, X_test, energy=0.995, random_state=SEED)
    measures = owls.alpha_from_variance(np.var(x_train, axis=0))   # symmetric Jacobi parameters matching each variance
    weights = pca.singular_values_ / pca.singular_values_[0]        # anisotropy weights
    print(f"d_in = {x_train.shape[1]} PCA modes")
    L = owls.index_set(x_train.shape[1], HC_K, "hc", weights, MAX_DEGREE, N_MAX)
    N = len(L)

    rel_errs, y_dims = [], []
    for i, alpha in enumerate(ALPHAS):
        y_train, y_test, compressor = encode_outputs(Y_train, Y_test, alpha=alpha, energy=0.999)
        model = owls.EmpiricalWLS(x_train, y_train, L, measures)
        model.fit(int(N * np.log(N)), optimal=True, rng=np.random.default_rng([SEED, i]))
        rel_errs.append(np.mean(model.compute_errors(x_test, y_test)[1]))
        y_dims.append(compressor.M)
        print(f"alpha = {alpha}: dim Y_h = {compressor.M}, mean relative H^alpha error {rel_errs[-1]:.3e}")
    np.save(os.path.join(DATA, "alphas.npy"), ALPHAS)
    np.save(os.path.join(DATA, "rel_Ha_errs.npy"), rel_errs)
    np.save(os.path.join(DATA, "Y_dims.npy"), y_dims)
    np.save(os.path.join(DATA, "Ns.npy"), [N] * len(ALPHAS))

In [ ]:
alphas = np.load(os.path.join(DATA, "alphas.npy"))
rel_errs = np.load(os.path.join(DATA, "rel_Ha_errs.npy"))
y_dims = np.load(os.path.join(DATA, "Y_dims.npy"))

fig_width, fig_height = paper_style(width=0.5, aspect=0.6, fontsize=7, legend_fontsize=5)
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout="constrained")
ax.plot(alphas, rel_errs, "ro--", label=r"Avg. Rel. $H^{\alpha}$ Error", markersize=3, linewidth=1)
ax.set(xlabel=r"$\alpha$", yscale="log")
ax.set_ylabel(r"Avg Relative $H^{\alpha}$ Error", color="r")
ax.tick_params(axis="y", labelcolor="r")
ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.plot(alphas, y_dims, "s-.", color="black", label=r"$\dim(\mathcal{Y}_{\alpha})$", markersize=3, linewidth=1)
ax2.legend(loc=4)
ax2.set(ylabel=r"$99.9\%$ Energy $\dim(\mathcal{Y}_{\alpha})$", yscale="log")
savefig(os.path.join(FIGS, "NS_err.pdf"))
plt.show()